Train: WQS0084

In [76]:
import sys
sys.path.insert(0, "../") # replace with path/to/project/root
from data import get_data, get_site_ids
import pandas as pd
import numpy as np


In [77]:
uid = "WQS0084"

data = get_data(site_uid=uid)

rain = data.rain
daily_avg_precip = (
    rain.groupby("date", as_index=False)["precip_in_1d"]
      .mean()
)

crops = data.crops.groupby("year", as_index=False).mean()

nitrogen_2017 = data.surplus[data.surplus['year'] == 2017]
SURPLUS_N_2017 = nitrogen_2017['surplus_kgha'].mean()


In [78]:
from data.water import aggregate_by_interval


water = aggregate_by_interval(site_uid=uid, value_col="nitrate_con", interval="1D", agg_func="max").to_frame()
water["date"] = water.index.date

water = water.reset_index(drop=True)

water["violation"] = (water.nitrate_con > 10).astype(int)
print("Violation counts:\n", water["violation"].value_counts())
water



Violation counts:
 violation
0    2600
1     264
Name: count, dtype: int64


,nitrate_con,date,violation
0,7.92,2018-07-25,0
1,7.92,2018-07-26,0
2,7.82,2018-07-27,0
3,7.69,2018-07-28,0
4,7.47,2018-07-29,0
...,...,...,...
2859,7.46,2026-05-23,0
2860,7.25,2026-05-24,0
2861,7.02,2026-05-25,0
2862,7.03,2026-05-26,0


In [79]:
water["date"] = pd.to_datetime(water["date"])
daily_avg_precip["date"] = pd.to_datetime(daily_avg_precip["date"])

df = water[["date", "violation"]].merge(
    daily_avg_precip,
    on="date",
    how="left"
)


In [80]:
df["rain_7d"]  = df["precip_in_1d"].rolling(7,  min_periods=1).sum()
df["rain_14d"] = df["precip_in_1d"].rolling(14, min_periods=1).sum()
df["rain_30d"] = df["precip_in_1d"].rolling(30, min_periods=1).sum()

# Crop data is yearly -> broadcast onto every day in that crop year.
# Note: if your crop year doesn't match the calendar year (e.g. planting
# season cutoffs), adjust this merge key accordingly.
df["year"] = df["date"].dt.year
df = df.merge(crops.drop(columns='node_id'), on="year", how="left")          # e.g. pct_corn, pct_soybean
 
# Surplus N is a single static estimate -> constant column.
# Be honest with yourself: this CANNOT explain year-to-year variation
# in violations, only a fixed basin-level "background risk" offset.
df["surplus_n_2017"] = SURPLUS_N_2017

# Optional interaction: leaching risk scales with both nutrient
# surplus and water flux through the soil
df["surplus_x_rain30"] = df["surplus_n_2017"] * df["rain_30d"]
 
df = df.dropna(subset=["violation"]).sort_values("date").reset_index(drop=True)

df


,date,violation,precip_in_1d,rain_7d,rain_14d,rain_30d,year,Alfalfa,Corn,Fallow,Hay_Pasture,Nonag,Other,Small_Grains,Soybeans,surplus_n_2017,surplus_x_rain30
0,2018-07-25,0,0.214848,0.214848,0.214848,0.214848,2018,231.050505,9175.390572,8.942761,1827.824916,2915.777778,33.026936,35.063973,6401.936027,80.879331,17.376802
1,2018-07-26,0,0.000707,0.215556,0.215556,0.215556,2018,231.050505,9175.390572,8.942761,1827.824916,2915.777778,33.026936,35.063973,6401.936027,80.879331,17.433989
2,2018-07-27,0,0.000000,0.215556,0.215556,0.215556,2018,231.050505,9175.390572,8.942761,1827.824916,2915.777778,33.026936,35.063973,6401.936027,80.879331,17.433989
3,2018-07-28,0,0.000067,0.215623,0.215623,0.215623,2018,231.050505,9175.390572,8.942761,1827.824916,2915.777778,33.026936,35.063973,6401.936027,80.879331,17.439436
4,2018-07-29,0,0.051919,0.267542,0.267542,0.267542,2018,231.050505,9175.390572,8.942761,1827.824916,2915.777778,33.026936,35.063973,6401.936027,80.879331,21.638625
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2859,2026-05-23,0,0.057710,0.986566,1.532357,2.024983,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80.879331,163.779285
2860,2026-05-24,0,0.031582,0.617778,1.563939,2.053266,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80.879331,166.066781
2861,2026-05-25,0,0.096364,0.298081,1.660303,2.149630,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80.879331,173.860607
2862,2026-05-26,0,0.489663,0.779596,2.135118,2.617239,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80.879331,211.680545


In [ ]:
cutoff = df["date"].quantile(0.8)          # last ~20% of days held out
train = df[df["date"] <= cutoff]
test  = df[df["date"] >  cutoff]
 
crop_cols = [c for c in crops.columns if c not in ["year", 'node_id']]
features = ["rain_7d", "rain_14d", "rain_30d", "surplus_x_rain30"] + crop_cols
 
X_train, y_train = train[features].fillna(0), train["violation"]
X_test,  y_test  = test[features].fillna(0),  test["violation"]
 
print(f"Train violations: {y_train.sum()} / {len(y_train)}")
print(f"Test  violations: {y_test.sum()} / {len(y_test)}")
if y_test.sum() == 0:
    print("WARNING: zero violations in the test window -- pick a different "
          "cutoff or use expanding-window CV instead of a single split.")


Train violations: 210 / 2291
Test  violations: 54 / 573


Use TimeSeriesSplit to split training data into 5 chunks

In [86]:
## Import TimeSeriesSplit
from sklearn.model_selection import TimeSeriesSplit

kfold = TimeSeriesSplit(n_splits = 5)

for train_index, test_index in kfold.split(train):
    print("TRAIN INDEX SIZE:", train_index.shape[0])
    print("TEST INDEX SIZE:", test_index.shape[0])
    print()
    print()

TRAIN INDEX SIZE: 386
TEST INDEX SIZE: 381


TRAIN INDEX SIZE: 767
TEST INDEX SIZE: 381


TRAIN INDEX SIZE: 1148
TEST INDEX SIZE: 381


TRAIN INDEX SIZE: 1529
TEST INDEX SIZE: 381


TRAIN INDEX SIZE: 1910
TEST INDEX SIZE: 381


